# Fraud Detection Model Building (Modularized)

This notebook demonstrates using the modular fraud detection framework with:
- Automated data loading and validation
- Feature engineering pipeline
- Class imbalance handling (SMOTE, class weighting)
- Comprehensive evaluation metrics (ROC AUC, PR AUC, confusion matrix)
- MLflow experiment tracking
- Model explainability (SHAP, LIME)

## 1. Load and Validate Data

Using the modular DataLoader to load datasets with built-in validation.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from scripts.data.loader import DataLoader
from scripts.features.engineering import FeatureEngineer, prepare_fraud_data, prepare_credit_data
from scripts.models.train import ModelTrainer, ModelFactory, ResamplingStrategy
from scripts.models.evaluate import ModelEvaluator, compare_models
from scripts.models.explainability import ModelExplainer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

print("✓ All modules imported successfully!")

In [ ]:
loader = DataLoader(data_dir='../data')
fraud_data, credit_data = loader.load_both_datasets(validate=True)

# fraud_data_combined.csv already has 'country'; the raw Fraud_Data.csv needs the IP lookup
ip_country = None if 'country' in fraud_data.columns else loader.load_ip_country_data()

print(f"\n✓ Fraud dataset loaded: {fraud_data.shape}")
print(f"✓ Credit dataset loaded: {credit_data.shape}")

## 2. Prepare Features

`prepare_fraud_data` adds country, hour_of_day, day_of_week, transaction_count (per device), time_since_last_transaction and signup_purchase_diff, and drops identifiers and raw timestamps. `FeatureEngineer` then encodes and scales, fitting on the training split only.

In [ ]:

X_fraud, y_fraud = prepare_fraud_data(fraud_data, target_col='class', ip_country_df=ip_country)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_fraud, y_fraud, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_fraud
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")
print(f"\nClass distribution (training):")
print(y_train.value_counts())
print(f"\nClass imbalance ratio: {y_train.value_counts()[0] / y_train.value_counts()[1]:.2f}:1")

In [ ]:

engineer = FeatureEngineer()
X_train_prep, X_test_prep = engineer.prepare_features(X_train, X_test, scale=True, encode=True)

print(f"✓ Feature engineering complete!")
print(f"Prepared training set: {X_train_prep.shape}")
print(f"Prepared test set: {X_test_prep.shape}")
print(f"Feature names: {engineer.feature_names[:10]}...") 

## 3. Train Models with Class Imbalance Handling

Training multiple models with different configurations:
- **Class weighting**: `class_weight='balanced'`
- **SMOTE**: Synthetic Minority Over-sampling Technique
- **SMOTETomek**: SMOTE + Tomek links for better boundaries

In [ ]:
# Initialize trainer with MLflow
trainer = ModelTrainer(experiment_name="fraud_detection_notebook")

# Define model configurations to test
model_configs = {
    'rf_balanced': {
        'model_name': 'random_forest',
        'resampling': 'none',
        'use_class_weight': True,
        'params': {'n_estimators': 100, 'max_depth': 10, 'random_state': 42}
    },
    'rf_smote': {
        'model_name': 'random_forest',
        'resampling': 'smote',
        'use_class_weight': False,
        'params': {'n_estimators': 100, 'max_depth': 10, 'random_state': 42}
    },
    'gb_balanced': {
        'model_name': 'gradient_boosting',
        'resampling': 'none',
        'use_class_weight': False,  # GB doesn't support class_weight
        'params': {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 5, 'random_state': 42}
    },
    'lr_smote_tomek': {
        'model_name': 'logistic_regression',
        'resampling': 'smote_tomek',
        'use_class_weight': False,
        'params': {'C': 1.0, 'random_state': 42}
    }
}

print("Training models with different configurations...")
print("This may take a few minutes...\n")

In [ ]:
# Train all models
results = trainer.train_multiple_models(
    model_configs=model_configs,
    X_train=X_train_prep,
    y_train=y_train,
    X_test=X_test_prep,
    y_test=y_test,
    output_dir='../models'
)

print("\n✓ All models trained successfully!")

## 4. Compare Model Performance

Comprehensive metrics comparison including:
- **Accuracy, Precision, Recall, F1-Score**
- **ROC AUC**: Area under ROC curve (overall discriminative power)
- **PR AUC**: Area under Precision-Recall curve (critical for imbalanced data)

In [ ]:
# Extract metrics for comparison
metrics_dict = {name: metrics for name, (_, metrics) in results.items()}

# Compare models
comparison_df = compare_models(metrics_dict)
print("\nModel Comparison:")
print(comparison_df)

# Visualize comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics_to_plot = ['precision', 'recall', 'f1_score', 'roc_auc']
titles = ['Precision', 'Recall', 'F1 Score', 'ROC AUC']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    ax = axes[idx // 2, idx % 2]
    comparison_df[metric].plot(kind='barh', ax=ax, color='skyblue')
    ax.set_title(f'{title} Comparison')
    ax.set_xlabel(title)
    ax.set_xlim([0, 1])
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Find best model
best_model_name = comparison_df['f1_score'].idxmax()
print(f"\n✓ Best model by F1-Score: {best_model_name}")
print(f"  F1: {comparison_df.loc[best_model_name, 'f1_score']:.4f}")
print(f"  Recall: {comparison_df.loc[best_model_name, 'recall']:.4f}")
print(f"  Precision: {comparison_df.loc[best_model_name, 'precision']:.4f}")

## 5. Detailed Evaluation of Best Model

Generate comprehensive evaluation reports including:
- Confusion Matrix
- ROC Curve
- Precision-Recall Curve
- Optimal Threshold Analysis

In [ ]:
# Get best model
best_model, best_metrics = results[best_model_name]

# Make predictions
y_pred = best_model.predict(X_test_prep)
y_pred_proba = best_model.predict_proba(X_test_prep)[:, 1]

# Create evaluator
evaluator = ModelEvaluator(best_model_name)

# Plot confusion matrix
fig = evaluator.plot_confusion_matrix(y_test, y_pred, figsize=(8, 6))
plt.show()

# Plot ROC curve
fig = evaluator.plot_roc_curve(y_test, y_pred_proba, figsize=(8, 6))
plt.show()

# Plot PR curve
fig = evaluator.plot_precision_recall_curve(y_test, y_pred_proba, figsize=(8, 6))
plt.show()

In [ ]:
# Find optimal threshold for business requirements
print("Threshold Analysis:")
print("="*60)

# F1-optimized threshold
optimal_f1 = evaluator.find_optimal_threshold(y_test, y_pred_proba, metric='f1')

# Precision-targeted threshold (e.g., 85% precision requirement)
precision_target = evaluator.evaluate_at_precision_target(y_test, y_pred_proba, target_precision=0.85)

print("\nSummary:")
print(f"Default threshold (0.5): Recall = {best_metrics['recall']:.4f}, Precision = {best_metrics['precision']:.4f}")
print(f"F1-optimized threshold ({optimal_f1['threshold']:.3f}): Recall = {optimal_f1['recall']:.4f}, Precision = {optimal_f1['precision']:.4f}")
print(f"Precision-targeted ({precision_target['threshold']:.3f}): Recall = {precision_target['recall']:.4f}, Precision = {precision_target['precision']:.4f}")

In [ ]:
# Save the best model with its fitted FeatureEngineer so the API can score raw feature rows
sys.path.insert(0, os.path.abspath('..'))
from src.models.predict import ModelRegistry

registry = ModelRegistry(models_dir='../models')
registry.save_model(
    best_model,
    'production_model.pkl',
    transformer=engineer,
    metadata={
        'name': best_model_name,
        'input_features': list(X_train.columns),
        'f1_optimal_threshold': float(optimal_f1['threshold']),
        'metrics': {k: float(v) for k, v in best_metrics.items() if isinstance(v, (int, float, np.integer, np.floating))},
    },
)

## 6. Model Explainability (SHAP & LIME)

Understanding **why** the model makes predictions:

In [ ]:
# Initialize explainer
explainer = ModelExplainer(best_model, X_train_prep)

# SHAP Summary Plot (global feature importance)
print("Generating SHAP summary plot...")
shap_values = explainer.explain_with_shap(
    X_test_prep.sample(min(100, len(X_test_prep))),
    plot_type='summary',
    max_display=15
)

# Feature importance comparison
print("\nFeature Importance:")
importance_df = explainer.get_feature_importance(method='shap', top_n=15)
explainer.plot_feature_importance(method='shap', top_n=15, figsize=(10, 8))

In [ ]:
# LIME: Explain individual fraud predictions
print("Generating LIME explanations for sample fraud cases...")

# Find a few fraud cases
fraud_indices = y_test[y_test == 1].index[:3]

for i, idx in enumerate(fraud_indices):
    instance = X_test_prep.loc[idx].values
    print(f"\nExplaining fraud case {i+1}:")
    explanation = explainer.explain_with_lime(instance, instance_index=i, num_features=10)